In [20]:
##### Run raster model using country average intensities 

import pandas as pd
import numpy as np
import rasterio
from pathlib import Path

In [21]:
##### SET-UP

### Set directories 
cd = Path.cwd().parent 
out_dir = cd / "Results" / "Raster_models" / "country_avg_model"
out_dir.mkdir(parents=True, exist_ok=True)

### Import data 

# Production data (raster)
production_path = f'{cd}/Data/total_production_tonnes_2020.tif'

# Intensities 
intensities = pd.read_csv(f'{cd}/Data/national_data.csv')

# crosswalks
country_crosswalk = pd.read_parquet(f'{cd}/Data/pixel_country_crosswalk.parquet')
capital_sub_crosswalk = pd.read_parquet(f'{cd}/Data/pixel_sub_capital_crosswalk.parquet')
labor_sub_crosswalk = pd.read_parquet(f'{cd}/Data/pixel_sub_labor_crosswalk.parquet')
country_names = pd.read_csv(f'{cd}/Data/country_names.csv', encoding="cp1252")

### Set paths 
capital_path = f"{cd}/Results/Raster_model/country_avg_model/capital_USD.tif"
labor_path = f"{cd}/Results/Raster_model/country_avg_model/jobs.tif"

In [22]:
##### PREP COUNTRY INTENSITIES 

# prep merge
iso_to_gid = (
    country_names[["ISO3", "SHP_code"]]
    .drop_duplicates(subset="ISO3")
    .rename(columns={"SHP_code": "GID_0"})
)
intensities = intensities.merge(iso_to_gid, on="ISO3", how="left")
intensities = intensities.dropna(subset=["GID_0"])

# National totals in absolute units 
intensities["national_total_capital_USD"] = intensities["ag_capital_stock_mil_USD_nominal"] * 1e6
intensities["national_total_jobs"] = intensities["ag_labor_thousands_2020"] * 1e3

intensities = intensities[[
    "GID_0",
    "capital_intensity_USD_per_tonne",
    "labor_intensity_jobs_per_tonne",
    "national_total_capital_USD",
    "national_total_jobs",
]]

In [23]:
##### LOAD PRODUCTION RASTER 

with rasterio.open(production_path) as src:
    prod = src.read(1).astype("float32")
    nodata = src.nodata
    transform = src.transform
    crs = src.crs
    height, width = src.height, src.width

valid = np.isfinite(prod)
if nodata is not None:
    valid &= (prod != nodata)

rows, cols = np.nonzero(valid)
xs, ys = rasterio.transform.xy(transform, rows, cols)  # pixel centres

pix = pd.DataFrame({
    "row": rows,
    "col": cols,
    "x": np.asarray(xs),
    "y": np.asarray(ys),
    "production_t": prod[rows, cols],
})
del prod, valid


def save_raster(values, path):
    arr = np.full((height, width), np.nan, dtype="float32")
    arr[pix["row"].to_numpy(), pix["col"].to_numpy()] = np.asarray(values, dtype="float32")
    with rasterio.open(
        path, "w", driver="GTiff", height=height, width=width, count=1,
        dtype="float32", crs=crs, transform=transform, nodata=np.nan, compress="lzw",
    ) as dst:
        dst.write(arr, 1)

In [24]:
##### ATTACH COUNTRY AVERAGE INTENSITIES TO EACH PIXEL, THEN MULTIPLY BY PRODUCTION

pix = pix.merge(country_crosswalk[["x", "y", "GID_0"]], on=["x", "y"], how="left")
pix = pix.merge(intensities, on="GID_0", how="left")

pix["capital_USD"] = pix["production_t"] * pix["capital_intensity_USD_per_tonne"]
pix["jobs"] = pix["production_t"] * pix["labor_intensity_jobs_per_tonne"]

save_raster(pix["capital_USD"], f"{out_dir}/capital_USD.tif")
save_raster(pix["jobs"], f"{out_dir}/jobs.tif")

##### SAVE SUB-NATIONAL AND NATIONAL PREDICTED TOTALS (before rescaling)

cap_sub = pix[["x", "y", "capital_USD"]].merge(capital_sub_crosswalk[["x", "y", "PROJ_ID"]], on=["x", "y"], how="left")
lab_sub = pix[["x", "y", "jobs"]].merge(labor_sub_crosswalk[["x", "y", "PROJ_ID"]], on=["x", "y"], how="left")

cap_sub.groupby("PROJ_ID", as_index=False)["capital_USD"].sum().to_csv(
    f"{out_dir}/capital_subnational_agg.csv", index=False)
lab_sub.groupby("PROJ_ID", as_index=False)["jobs"].sum().to_csv(
    f"{out_dir}/labor_subnational_agg.csv", index=False)
del cap_sub, lab_sub

nat = pix.groupby("GID_0", as_index=False).agg(
    predicted_capital_USD=("capital_USD", "sum"),
    predicted_jobs=("jobs", "sum"),
)
nat.to_csv(f"{out_dir}/national_predicted_agg.csv", index=False)

In [25]:
##### RESCALE TO NATIONAL TOTALS

# Step 1: get national targets
nat = nat.merge(
    intensities[["GID_0", "national_total_capital_USD", "national_total_jobs"]],
    on="GID_0", how="left",
)

# Step 2: calculate rescaling factors 
nat["capital_nat_scale"] = (nat["national_total_capital_USD"] / nat["predicted_capital_USD"]).replace([np.inf, -np.inf], np.nan)
nat["labor_nat_scale"] = (nat["national_total_jobs"] / nat["predicted_jobs"]).replace([np.inf, -np.inf], np.nan)

pix = pix.merge(nat[["GID_0", "capital_nat_scale", "labor_nat_scale"]], on="GID_0", how="left")

# Step 3: apply
pix["rescaled_capital_USD"] = pix["capital_USD"] * pix["capital_nat_scale"]
pix["rescaled_jobs"] = pix["jobs"] * pix["labor_nat_scale"]

In [26]:
##### SANITY CHECKS

# Compare rescaled pixel sums to national targets
check = pix.groupby("GID_0").agg(
    rescaled_capital=("rescaled_capital_USD", "sum"),
    rescaled_jobs=("rescaled_jobs", "sum"),
).join(nat.set_index("GID_0")[["national_total_capital_USD", "national_total_jobs",
                               "capital_nat_scale", "labor_nat_scale"]])

no_scale = check[check["capital_nat_scale"].isna() | check["labor_nat_scale"].isna()]
ok = check.drop(index=no_scale.index).dropna(subset=["national_total_capital_USD", "national_total_jobs"])

cap_pct = ok["rescaled_capital"].sum() / ok["national_total_capital_USD"].sum() * 100
lab_pct = ok["rescaled_jobs"].sum() / ok["national_total_jobs"].sum() * 100

print("Sum of rescaled pixels as % of sum of national targets (rescaled countries only; should be 100):")
print(f"Capital: {cap_pct:.2f}")
print(f"Labor: {lab_pct:.2f}")

Sum of rescaled pixels as % of sum of national targets (rescaled countries only; should be 100):
Capital: 100.00
Labor: 100.00


In [27]:
##### SAVE FINAL OUTPUTS

save_raster(pix["rescaled_capital_USD"], f"{out_dir}/rescaled_capital_USD.tif")
print(f"Saved rescaled_capital_USD -> {out_dir}/rescaled_capital_USD.tif")
save_raster(pix["rescaled_jobs"], f"{out_dir}/rescaled_jobs.tif")
print(f"Saved rescaled_jobs -> {out_dir}/rescaled_jobs.tif")

Saved rescaled_capital_USD -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/country_avg_model/rescaled_capital_USD.tif
Saved rescaled_jobs -> /Users/carinamanitius/Documents/GitHub/Gridded_Agricultural_Capital_Labor/Results/Raster_models/country_avg_model/rescaled_jobs.tif
